# 01 — Data check

Run this **before** training. It answers three questions:

1. Did the conversion to YOLO format work?
2. How many people are in a typical image?
3. Do the boxes actually sit on the people?

If something looks wrong here, training will waste hours. Check first.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

DATA = Path('../data')
SPLIT = 'train'

images_dir = DATA / SPLIT / 'images'
labels_dir = DATA / SPLIT / 'labels'

images = sorted(p for p in images_dir.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})
labels = sorted(labels_dir.glob('*.txt'))

print(f'images: {len(images)}')
print(f'labels: {len(labels)}')

## Every image needs a matching label file

A mismatch here is the most common conversion bug.

In [ ]:
image_stems = {p.stem for p in images}
label_stems = {p.stem for p in labels}

missing_labels = image_stems - label_stems
orphan_labels = label_stems - image_stems

print(f'images with no label file : {len(missing_labels)}')
print(f'label files with no image : {len(orphan_labels)}')

if missing_labels:
    print('\nexamples:', list(missing_labels)[:5])

## How many people per image?

CrowdHuman averages roughly 23 people per image. If your average comes out near
1 or 2, the conversion probably kept only one box per image.

In [ ]:
counts = []
for label_file in labels:
    lines = [ln for ln in label_file.read_text().splitlines() if ln.strip()]
    counts.append(len(lines))

if counts:
    counts_sorted = sorted(counts)
    print(f'total people   : {sum(counts):,}')
    print(f'mean per image : {sum(counts) / len(counts):.1f}')
    print(f'median         : {counts_sorted[len(counts_sorted) // 2]}')
    print(f'min / max      : {min(counts)} / {max(counts)}')
    print(f'empty images   : {sum(1 for c in counts if c == 0)}')

    plt.figure(figsize=(8, 4))
    plt.hist(counts, bins=40)
    plt.xlabel('people per image')
    plt.ylabel('number of images')
    plt.title(f'People per image — {SPLIT} split')
    plt.show()

## Check the box values are valid

YOLO expects `class x_center y_center width height`, with all four numbers
between 0 and 1. Anything outside that range means the conversion used pixels
instead of relative values.

In [ ]:
bad_lines = 0
checked = 0

for label_file in labels[:500]:
    for line in label_file.read_text().splitlines():
        parts = line.split()
        if not parts:
            continue
        checked += 1
        if len(parts) != 5:
            bad_lines += 1
            continue
        values = [float(v) for v in parts[1:]]
        if any(v < 0 or v > 1 for v in values):
            bad_lines += 1

print(f'lines checked : {checked}')
print(f'invalid lines : {bad_lines}')
print('OK' if bad_lines == 0 else 'Fix the conversion before training.')

## Look at the boxes yourself

Numbers can look fine while the boxes sit in the wrong place. Always look.

In [ ]:
import matplotlib.patches as patches

SAMPLES = 3

for image_path in images[:SAMPLES]:
    label_file = labels_dir / (image_path.stem + '.txt')
    if not label_file.exists():
        continue

    image = Image.open(image_path)
    width, height = image.size

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.imshow(image)

    people = 0
    for line in label_file.read_text().splitlines():
        parts = line.split()
        if len(parts) != 5:
            continue
        _, xc, yc, w, h = (float(v) for v in parts)
        x = (xc - w / 2) * width
        y = (yc - h / 2) * height
        ax.add_patch(patches.Rectangle((x, y), w * width, h * height,
                                       fill=False, edgecolor='lime', linewidth=1.5))
        people += 1

    ax.set_title(f'{image_path.name} — {people} people')
    ax.axis('off')
    plt.show()

## What to write down

Note these for your report — they belong in the data section:

- number of images in train and val
- mean and median people per image
- how many images are empty
- anything odd you noticed in the sample images

Then run the baseline evaluation **before** training anything:

```bash
cd backend
python evaluate.py --model baseline --name baseline --data crowdhuman.yaml
```